# 1. Modelo em TensorFlow/Keras

### Aqui treinamos nossa CNN de classificação binária:

### - Classe 0: Vaga Livre

### - Classe 1: Vaga Ocupada

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Entrada padronizada para 64x64 em RGB (ideal para economizar SRAM no ESP32)
IMG_SHAPE = (64, 64, 3)

model = models.Sequential([
    layers.Input(shape=IMG_SHAPE),
    layers.Rescaling(1./255),

    # Bloco Convolucional 1
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Bloco Convolucional 2
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Classificador
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Saída Binária
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## 2. Quantização para INT8 (TensorFlow Lite)

### Para o ESP32 processar a rede usando extensões vetoriais de alta velocidade e consumir pouca RAM, convertemos o modelo para Full INT8 Quantization:

In [6]:
import numpy as np
import tensorflow as tf

# Gerador sintético de calibração INT8
def representative_data_gen():
    for _ in range(100):
        # Gera dados aleatórios no formato (1, 64, 64, 3) com valores entre 0 e 255
        data = tf.random.uniform([1, 64, 64, 3], minval=0.0, maxval=255.0, dtype=tf.float32)
        yield [data]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Executa a conversão e quantização
tflite_quant_model = converter.convert()

# Salva o modelo quantizado
with open('smart_parking_model_int8.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print("Modelo quantizado gerado com sucesso!")

Saved artifact at '/tmp/tmphrdr4scf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132790907761488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907762448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907761680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907762640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907763408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907763600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907762256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132790907763792: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Modelo quantizado gerado com sucesso!


# 3. Converter o Modelo .tflite para C-Array (model_data.h)

In [7]:
# Converte o arquivo .tflite para um header C (.h)
!xxd -i smart_parking_model_int8.tflite > model_data.h